# 02 - Exploratory Data Analysis (EDA)
**ZHAW AI-Applications: Skin Lesion Classifier**

Analyzes the HAM10000 dataset: class distribution, image statistics, metadata, and correlations.

In [ ]:
import sys, os
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

PROC_DIR = Path('../data/processed')
RAW_DIR  = Path('../data/raw')
print('EDA setup complete')

## 1. Class Distribution

In [ ]:
train = pd.read_csv(PROC_DIR / 'train.csv')
val   = pd.read_csv(PROC_DIR / 'val.csv')
test  = pd.read_csv(PROC_DIR / 'test.csv')
df    = pd.concat([train, val, test])

CLASS_NAMES = {'mel': 'Melanoma', 'nv': 'Melanocytic nevi', 'bcc': 'Basal cell carcinoma',
               'akiec': 'Actinic keratoses', 'bkl': 'Benign keratosis', 'df': 'Dermatofibroma', 'vasc': 'Vascular lesions'}

counts = df['dx'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar([CLASS_NAMES[k] for k in counts.index], counts.values, color=sns.color_palette('husl', 7))
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha='right')
axes[0].set_title('Class Distribution (absolute)')
axes[0].set_ylabel('Count')

axes[1].pie(counts.values, labels=[CLASS_NAMES[k] for k in counts.index], autopct='%1.1f%%', colors=sns.color_palette('husl', 7))
axes[1].set_title('Class Distribution (%)')

plt.suptitle('HAM10000 Class Distribution', fontsize=14)
plt.tight_layout()
plt.savefig('../docs/screenshots/class_distribution.png', dpi=150)
plt.show()

print('Class imbalance ratio (max/min):', round(counts.max()/counts.min(), 1), 'x')

## 2. Metadata Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Age distribution per class
for cls in df['dx'].unique():
    subset = df[df['dx']==cls]['age'].dropna()
    axes[0][0].hist(subset, alpha=0.5, label=cls, bins=20)
axes[0][0].set_title('Age Distribution per Class')
axes[0][0].set_xlabel('Age')
axes[0][0].legend(fontsize=7)

# Sex distribution
sex_ct = pd.crosstab(df['dx'], df['sex'])
sex_ct.plot(kind='bar', ax=axes[0][1], colormap='Set2')
axes[0][1].set_title('Sex Distribution per Class')
axes[0][1].tick_params(axis='x', rotation=30)

# Localization heatmap
loc_ct = pd.crosstab(df['dx'], df['localization'])
sns.heatmap(loc_ct, ax=axes[1][0], cmap='Blues', annot=False)
axes[1][0].set_title('Localization per Class')

# Diagnosis type
dx_ct = pd.crosstab(df['dx'], df['dx_type'])
dx_ct.plot(kind='bar', ax=axes[1][1], colormap='Set3')
axes[1][1].set_title('Diagnosis Type per Class')
axes[1][1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../docs/screenshots/metadata_analysis.png', dpi=150)
plt.show()

## 3. Image Statistics

In [ ]:
# Sample 200 images for statistics
sample = df.sample(200, random_state=42)
sizes, means, stds = [], [], []

for _, row in sample.iterrows():
    try:
        img = np.array(Image.open(row['image_path']).convert('RGB'))
        sizes.append(img.shape[:2])
        means.append(img.mean(axis=(0,1)))
        stds.append(img.std(axis=(0,1)))
    except:
        pass

means = np.array(means)
print(f'Mean pixel values (RGB): {means.mean(axis=0).round(2)}')
print(f'Std pixel values (RGB):  {np.array(stds).mean(axis=0).round(2)}')
print(f'Image sizes: {set(sizes)}')

## 4. Train/Val/Test Split Summary

In [ ]:
for name, split in [('Train', train), ('Val', val), ('Test', test)]:
    print(f'{name:5}: {len(split):5} samples')
    print(split['dx'].value_counts().to_string())
    print()

print('Total:', len(df))

## 5. Key EDA Findings

- **Class Imbalance**: nv (melanocytic nevi) is heavily overrepresented (~67% of data)
- **Age**: Most lesions occur in 40-60 age range
- **Sex**: Slight male majority for most lesion types
- **Localization**: Back and lower extremity most common
- **Image Size**: All images are 600x450 px (dermatoscopy)
- **Mitigation**: Class-weighted loss function in training